# Non-Financial Time-Series Forecasting Case Study (SARIMAX)

In [ ]:
!pip install pmdarima --quiet

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["PYTHONWARNINGS"] = "ignore"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pmdarima as pm

from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing

RANDOM_STATE = 42
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

## 1. Load data and perform a minimal global audit

In [ ]:
path = "non_financial_ts_sarimax_case.csv"
df_raw = pd.read_csv(path, parse_dates=["date"])

print("Raw shape:", df_raw.shape)
print("Date range:", df_raw["date"].min(), "to", df_raw["date"].max())
print("Duplicate rows:", int(df_raw.duplicated().sum()))
print("Missing values by column:")
display(df_raw.isna().sum().to_frame("missing_count").T)
print("Dtypes:")
display(df_raw.dtypes.astype(str).to_frame("dtype").T)

In [ ]:
summary_table = pd.DataFrame({
    "min": df_raw.drop(columns=["date"]).min(numeric_only=True),
    "max": df_raw.drop(columns=["date"]).max(numeric_only=True),
    "mean": df_raw.drop(columns=["date"]).mean(numeric_only=True),
    "std": df_raw.drop(columns=["date"]).std(numeric_only=True),
})
display(summary_table)

This is a monthly series of 144 observations (12 years) with 6 potential covariates. One duplicate row, 4 missing values in search_interest_index — both minor.

Key observation: this is a non-financial series, so I am not starting from the prior that the level is stationary. Trend, seasonality and structural breaks are likely first-order objects rather than nuisances.

Intentionally simple cleaning: sort chronologically, drop the one duplicate, linearly interpolate the 4 missing search_interest values. No smoothing — the forecasting problem is precisely about preserving level and seasonal structure.

In [ ]:
df = df_raw.sort_values("date").drop_duplicates().copy()
df["search_interest_index"] = df["search_interest_index"].interpolate(limit_direction="both")
df = df.set_index("date")
df.index.freq = "MS"
print("Post-cleaning shape:", df.shape)

## 2. Reserve a final holdout before detailed EDA

In [ ]:
holdout_size = 24
dev_df = df.iloc[:-holdout_size].copy()
holdout_df = df.iloc[-holdout_size:].copy()

display(pd.DataFrame({
    "sample": ["development", "final_holdout"],
    "rows": [len(dev_df), len(holdout_df)],
    "start_date": [dev_df.index.min(), holdout_df.index.min()],
    "end_date": [dev_df.index.max(), holdout_df.index.max()],
}))

## 3. EDA on the development sample only

### 3.1 Level series and potential drivers

In [ ]:
%matplotlib inline

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(dev_df.index, dev_df["passengers_thousands"], label="passengers")
axes[0].set_title("Passenger series")
axes[1].plot(dev_df.index, dev_df["search_interest_index"], label="search_interest")
axes[1].plot(dev_df.index, dev_df["ticket_price_index"], label="ticket_price_index")
axes[1].legend()
axes[1].set_title("Potential exogenous drivers")
plt.tight_layout()
plt.show();

Three features are immediately visible: (1) a clear upward trend, (2) strong annual seasonality with amplitude that grows with the level, and (3) a sharp structural break around 2020. The growing seasonal amplitude suggests **multiplicative** seasonality — I will test both additive and multiplicative specifications.

### 3.2 Seasonal decomposition

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

decomp_add = sm.tsa.seasonal_decompose(dev_df["passengers_thousands"], model="additive", period=12)
decomp_mult = sm.tsa.seasonal_decompose(dev_df["passengers_thousands"], model="multiplicative", period=12)

axes[0].plot(decomp_add.resid.dropna(), label="additive residuals")
axes[0].set_title("Additive decomposition residuals")
axes[0].axhline(0, color="black", lw=0.5)

axes[1].plot(decomp_mult.resid.dropna(), label="multiplicative residuals")
axes[1].set_title("Multiplicative decomposition residuals")
axes[1].axhline(1, color="black", lw=0.5)

plt.tight_layout()
plt.show()

print("Additive residual std:", round(decomp_add.resid.dropna().std(), 2))
print("Multiplicative residual std:", round(decomp_mult.resid.dropna().std(), 4))

Two arguments point in different directions. The **additive** decomposition has a lower relative residual std (12.43 / 350 ≈ 3.6%) compared to the **multiplicative** (0.060 / 1.0 = 6.0%), meaning it captures slightly more variance in aggregate.

However, the multiplicative residuals are visibly more **homoscedastic** — their variance stays roughly constant over time, whereas the additive residuals fan out as the level grows. For forecasting, constant-variance residuals matter more than a marginally lower global std, because they produce confidence intervals with more uniform coverage across the forecast horizon.

Combined with the visual evidence from §3.1 (seasonal amplitude clearly grows with the level), this supports treating seasonality as multiplicative — either via Holt-Winters multiplicative or via a log-transform of the SARIMAX target. Time permitting, it would be worth testing SARIMAX on both `passengers` and `log(passengers)` to quantify the impact; the log-transform would convert the multiplicative structure into an additive one that ARIMA handles natively.

### 3.3 Stationarity tests

In [ ]:
adf_level = adfuller(dev_df["passengers_thousands"])
adf_diff = adfuller(dev_df["passengers_thousands"].diff().dropna())
adf_diff_seasonal = adfuller(dev_df["passengers_thousands"].diff().diff(12).dropna())

display(pd.DataFrame({
    "series": ["level", "first_difference (d=1)", "first_diff + seasonal_diff (d=1, D=1)"],
    "adf_stat": [adf_level[0], adf_diff[0], adf_diff_seasonal[0]],
    "p_value": [adf_level[1], adf_diff[1], adf_diff_seasonal[1]],
}))

--> ADF on levels fails to reject H0 (p=0.999). First differencing alone is marginal (p=0.122) — the seasonal component adds persistent autocorrelation that a single difference does not remove. After both regular and seasonal differencing, ADF rejects decisively (p=0.0002), confirming that **(d=1, D=1)** is the right differencing specification.

### 3.4 ACF/PACF of differenced series

In [ ]:
# Apply both regular and seasonal differencing before inspecting ACF/PACF
y = dev_df["passengers_thousands"]
y_diff = y.diff().dropna()  # d=1
y_diff_seasonal = y_diff.diff(12).dropna()  # D=1, s=12

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
plot_acf(y_diff, lags=36, ax=axes[0, 0], title="ACF after d=1")
plot_pacf(y_diff, lags=24, ax=axes[0, 1], title="PACF after d=1")
plot_acf(y_diff_seasonal, lags=36, ax=axes[1, 0], title="ACF after d=1, D=1")
plot_pacf(y_diff_seasonal, lags=24, ax=axes[1, 1], title="PACF after d=1, D=1")
plt.tight_layout()
plt.show()

After single differencing (top row), strong spikes remain at lags 12, 24, 36 — confirming the need for seasonal differencing. After both d=1 and D=1 (bottom row), significant spikes persist at lag 1 and lag 12 in both ACF and PACF, confirming that there is temporal structure beyond what differencing alone removes.

Interpreting ACF/PACF for seasonal ARIMA is notoriously subjective — different analysts could read these same plots as (0,1,1)(0,1,1,12), (1,1,1)(1,1,1,12), or other specifications. Rather than committing to a manual order, I use `pmdarima.auto_arima` in §4 to systematically select the ARIMA and seasonal orders via AIC minimization. The ACF/PACF analysis confirms that the series has enough residual structure to justify a SARIMA model, which is the key diagnostic conclusion.

### 3.5 Exogenous variable inspection

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].scatter(dev_df["search_interest_index"], dev_df["passengers_thousands"], alpha=0.5, s=15)
axes[0,0].set_xlabel("search_interest_index"); axes[0,0].set_ylabel("passengers")
axes[0,0].set_title("Search interest vs passengers")

axes[0,1].scatter(dev_df["ticket_price_index"], dev_df["passengers_thousands"], alpha=0.5, s=15)
axes[0,1].set_xlabel("ticket_price_index"); axes[0,1].set_ylabel("passengers")
axes[0,1].set_title("Ticket price vs passengers")

axes[1,0].scatter(dev_df["weather_severity_index"], dev_df["passengers_thousands"], alpha=0.5, s=15)
axes[1,0].set_xlabel("weather_severity_index"); axes[1,0].set_ylabel("passengers")
axes[1,0].set_title("Weather severity vs passengers")

# Shock/strike timing
axes[1,1].plot(dev_df.index, dev_df["passengers_thousands"], alpha=0.5)
shock_dates = dev_df.loc[dev_df["shock_dummy"] == 1].index
strike_dates = dev_df.loc[dev_df["strike_dummy"] == 1].index
for d in shock_dates: axes[1,1].axvline(d, color="red", alpha=0.5, lw=1)
for d in strike_dates: axes[1,1].axvline(d, color="orange", alpha=0.5, lw=1)
axes[1,1].set_title("Shocks (red) and strikes (orange) on passengers")

plt.tight_layout()
plt.show()

# Correlations with passengers
corr_with_y = dev_df[["passengers_thousands"] + ["search_interest_index", "ticket_price_index", "weather_severity_index"]].corr()["passengers_thousands"].drop("passengers_thousands")
display(corr_with_y.to_frame("corr_with_passengers"))

Search interest (r=0.81) and ticket price (r=0.92) both show strong positive correlations with passengers — but these are almost certainly spurious, driven by shared upward trends. Weather severity (r=0.11) shows near-zero raw correlation. The shock dummy aligns precisely with the 2020 collapse.

The raw correlations are misleading. What matters is whether these variables explain variation *beyond* trend and seasonality.

### 3.6 Exogenous signal after removing trend: differenced residuals

The raw correlations between exogenous variables and passengers are dominated by shared trends. To assess whether the exogenous variables carry genuine predictive information, I difference both passengers and the continuous exogenous variables (first difference is sufficient for the exogenous since they do not have strong seasonality) and check the correlation of the residuals.

In [ ]:
# Difference passengers (d=1 + D=1) and exogenous (d=1 only, no seasonality to remove)
y_dd = dev_df["passengers_thousands"].diff().diff(12).dropna()
exog_continuous = ["search_interest_index", "ticket_price_index", "weather_severity_index"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
detrended_corrs = {}

for ax, col in zip(axes, exog_continuous):
    x_d = dev_df[col].diff().dropna()
    # Align indices
    common_idx = y_dd.index.intersection(x_d.index)
    y_aligned = y_dd.loc[common_idx]
    x_aligned = x_d.loc[common_idx]
    
    corr = y_aligned.corr(x_aligned)
    detrended_corrs[col] = round(corr, 3)
    
    ax.scatter(x_aligned, y_aligned, alpha=0.4, s=15)
    ax.set_xlabel(f"{col} (d=1)")
    ax.set_ylabel("passengers (d=1, D=1)")
    ax.set_title(f"r = {corr:.3f}")

plt.suptitle("Differenced passengers vs differenced exogenous", y=1.02)
plt.tight_layout()
plt.show()

print("Detrended correlations:")
for k, v in detrended_corrs.items():
    print(f"  {k}: {v}")

After detrending, the correlations drop dramatically: search_interest falls from 0.81 to 0.11, ticket_price from 0.92 to -0.15. Both are now negligible. Weather severity is the only continuous variable with a modest detrended correlation (0.17), suggesting it may carry some genuine incremental signal.

This confirms that the exogenous value in the SARIMAX comes primarily from the binary intervention variables (shock_dummy, strike_dummy), not from the continuous covariates. The continuous variables track the same trend as passengers but do not explain deviations from it.

### 3.7 Exogenous significance after detrending

In [ ]:
# OLS on double-differenced target vs differenced continuous exog + dummies in levels
sig_df = dev_df.copy()

sig_df["y_dd"] = sig_df["passengers_thousands"].diff().diff(12)
for col in ["search_interest_index", "ticket_price_index", "weather_severity_index"]:
    sig_df[f"{col}_d"] = sig_df[col].diff()

sig_df = sig_df.dropna()

ols_formula = (
    "y_dd ~ search_interest_index_d + ticket_price_index_d"
    " + weather_severity_index_d + strike_dummy + shock_dummy"
)
ols_exog = smf.ols(formula=ols_formula, data=sig_df).fit(cov_type="HC3")
print(ols_exog.summary())

This OLS tests whether the exogenous variables have any linear association with the target after removing trend and seasonality via double differencing (d=1, D=1). Continuous exogenous variables are also differenced (d=1) to avoid spurious regression from shared trends. The binary dummies are kept in levels since they are already stationary.

**No variable is significant** (all p > 0.10), and the F-test rejects nothing (p = 0.30, R² = 0.07). Even `shock_dummy` is non-significant (p = 0.54). This is not necessarily surprising: the OLS works on the *differenced* series, where a sustained level shift only appears as entry/exit spikes — a 0/1 dummy is not well suited to capture that in differences.

`ticket_price_index_d` comes closest (p = 0.10, coef = −7.57). `weather_severity_index_d` shows a positive but non-significant association (p = 0.16).

This OLS is intentionally conservative. A non-significant result here means no *obvious* marginal signal after detrending, but it does not rule out that these variables could be useful in a model that operates on levels rather than differences — the ARIMA error structure could amplify weak signals that a simple OLS on differences cannot detect.

## 4. Order selection and modeling setup

In [ ]:
exog_cols = ["search_interest_index", "ticket_price_index",
             "weather_severity_index", "strike_dummy", "shock_dummy"]

train_df = dev_df.copy()
covid_mask = (train_df.index >= "2020-03-01") & (train_df.index <= "2021-06-01")
train_no_covid = train_df[~covid_mask].copy()
final_holdout_df = holdout_df.copy()

print("Train:", len(train_df), "| Train excl. COVID:", len(train_no_covid),
      "| Holdout:", len(final_holdout_df))

I use `pmdarima.auto_arima` to systematically select the SARIMAX order via AIC minimization, with d=1 and D=1 fixed (validated by the stationarity tests in §3.3). The search is exhaustive (`stepwise=False`) to avoid the greedy path-dependency of the default stepwise approach. I run it on both the full training set and the COVID-excluded training set, to check whether COVID changes the optimal order.

In [ ]:
# ── auto_arima on full training set (exhaustive search) ──
auto_full = pm.auto_arima(
    train_df["passengers_thousands"], exogenous=train_df[exog_cols],
    seasonal=True, m=12, d=1, D=1,
    start_p=0, max_p=2, start_q=0, max_q=2,
    start_P=0, max_P=1, start_Q=0, max_Q=1,
    stepwise=False, n_jobs=-1,
    suppress_warnings=True, error_action="ignore"
)
print(f"auto_arima (full train): {auto_full.order} x {auto_full.seasonal_order}, AIC={auto_full.aic():.1f}")

# ── auto_arima on COVID-excluded training set (exhaustive search) ──
auto_no_covid = pm.auto_arima(
    train_no_covid["passengers_thousands"], exogenous=train_no_covid[exog_cols],
    seasonal=True, m=12, d=1, D=1,
    start_p=0, max_p=2, start_q=0, max_q=2,
    start_P=0, max_P=1, start_Q=0, max_Q=1,
    stepwise=False, n_jobs=-1,
    suppress_warnings=True, error_action="ignore"
)
print(f"auto_arima (excl. COVID): {auto_no_covid.order} x {auto_no_covid.seasonal_order}, AIC={auto_no_covid.aic():.1f}")

The exhaustive search selects **(0,1,1)(1,1,0,12)** on the full train (AIC = 931.6) and **(2,1,0)(0,1,1,12)** on the COVID-excluded train (AIC = 767.2). The structures are strikingly different: the full-train model uses MA(1) + seasonal AR(1), while the COVID-excluded model uses AR(2) + seasonal MA(1). This confirms that COVID does not just affect parameter values — it changes the model structure.

The AIC gap (931.6 vs 767.2) also reflects the noisier likelihood surface when COVID outliers are included. The COVID-excluded model achieves a much tighter fit on cleaner data.

## 5. Models — holdout evaluation

I fit four SARIMAX specifications (with/without COVID × with/without exogenous, using the respective auto_arima orders), one Holt-Winters multiplicative, and two naive baselines. The with/without exogenous comparison directly tests whether the covariates add value beyond the ARIMA structure — using the same order isolates the exogenous effect.

In [ ]:
results_all = []
fitted_models = {}
predictions = {}

# ── SARIMAX models: 2 training sets × 2 exog settings ──
model_specs = [
    ("SARIMAX with COVID + exog",    train_df,       auto_full.order,     auto_full.seasonal_order,     True),
    ("SARIMAX with COVID (no exog)", train_df,       auto_full.order,     auto_full.seasonal_order,     False),
    ("SARIMAX excl. COVID + exog",   train_no_covid, auto_no_covid.order, auto_no_covid.seasonal_order, True),
    ("SARIMAX excl. COVID (no exog)",train_no_covid, auto_no_covid.order, auto_no_covid.seasonal_order, False),
]

for label, train, order, seasonal_order, use_exog in model_specs:
    endog = train["passengers_thousands"]
    exog_train = train[exog_cols] if use_exog else None
    exog_test  = final_holdout_df[exog_cols] if use_exog else None

    mod = sm.tsa.statespace.SARIMAX(
        endog, exog=exog_train,
        order=order, seasonal_order=seasonal_order,
        enforce_stationarity=False, enforce_invertibility=False
    )
    fit_result = mod.fit(disp=False, maxiter=200, method="lbfgs")
    pred = fit_result.get_forecast(steps=len(final_holdout_df), exog=exog_test).predicted_mean.values

    results_all.append({
        "model": label, "order": f"{order}x{seasonal_order}",
        "mae": round(mean_absolute_error(final_holdout_df["passengers_thousands"], pred), 1),
        "rmse": round(rmse(final_holdout_df["passengers_thousands"], pred), 1),
        "aic": round(fit_result.aic, 1),
    })
    fitted_models[label] = fit_result
    predictions[label] = pred

# ── Holt-Winters multiplicative ──
hw = ExponentialSmoothing(
    train_df["passengers_thousands"],
    trend="mul", seasonal="mul", seasonal_periods=12,
    use_boxcox=False, initialization_method="estimated"
).fit(optimized=True)
pred_hw = hw.forecast(len(final_holdout_df)).values
results_all.append({"model": "Holt-Winters multiplicative", "order": "-",
    "mae": round(mean_absolute_error(final_holdout_df["passengers_thousands"], pred_hw), 1),
    "rmse": round(rmse(final_holdout_df["passengers_thousands"], pred_hw), 1),
    "aic": round(hw.aic, 1)})
predictions["Holt-Winters"] = pred_hw

# ── Baselines ──
full_ts = pd.concat([train_df["passengers_thousands"], final_holdout_df["passengers_thousands"]])
seasonal_naive_pred = full_ts.shift(12).loc[final_holdout_df.index].values
last_value_pred = np.repeat(train_df["passengers_thousands"].iloc[-1], len(final_holdout_df))

for lbl, pred_bl in [("seasonal_naive", seasonal_naive_pred), ("last_value", last_value_pred)]:
    results_all.append({"model": lbl, "order": "-",
        "mae": round(mean_absolute_error(final_holdout_df["passengers_thousands"], pred_bl), 1),
        "rmse": round(rmse(final_holdout_df["passengers_thousands"], pred_bl), 1),
        "aic": np.nan})
    predictions[lbl] = pred_bl

results_df = pd.DataFrame(results_all).sort_values("rmse").reset_index(drop=True)
display(results_df)

In [ ]:
# Holdout forecast visualization — zoomed on 2022-2026
zoom_start = pd.Timestamp("2022-01-01")
train_zoom = train_df[train_df.index >= zoom_start]

plt.figure(figsize=(12, 5))
plt.plot(train_zoom.index, train_zoom["passengers_thousands"], label="train", color="steelblue")
plt.plot(final_holdout_df.index, final_holdout_df["passengers_thousands"],
         label="actual holdout", color="black", linewidth=2)
plt.plot(final_holdout_df.index, seasonal_naive_pred,
         label="seasonal naive", linestyle="--", color="gray", alpha=0.6)
plt.plot(final_holdout_df.index, predictions["SARIMAX with COVID + exog"],
         label="SARIMAX with COVID + exog", color="tab:red", alpha=0.7)
plt.plot(final_holdout_df.index, predictions["SARIMAX excl. COVID + exog"],
         label="SARIMAX excl. COVID + exog", color="tab:purple", linewidth=2)
plt.plot(final_holdout_df.index, predictions["SARIMAX excl. COVID (no exog)"],
         label="SARIMAX excl. COVID (no exog)", color="tab:purple", linestyle="--", alpha=0.7)
plt.plot(final_holdout_df.index, pred_hw,
         label="Holt-Winters mult", color="tab:green", linestyle="--", alpha=0.7)
plt.legend(fontsize=8, loc="upper left")
plt.title("Holdout forecast comparison — zoomed 2022-2026")
plt.tight_layout()
plt.show()

The holdout comparison answers the key questions:

**Excluding COVID is the single biggest improvement.** SARIMAX excl. COVID + exog achieves RMSE 12.9 vs 22.4 for the with-COVID variant — a 42% reduction. The COVID period actively biases parameter estimates.

**Exogenous variables help in the excl. COVID model, but not in the with-COVID model.** Excl. COVID: RMSE drops from 17.6 (no exog) to 12.9 (with exog), a 27% improvement. With COVID: 22.4 vs 22.4 — no difference at all. This is surprising given that no individual exogenous variable is significant in the excl. COVID SARIMAX (§6 will confirm this). Collectively, the exogenous signal is real but diffuse — no single variable is strong enough to reach significance, but together they improve forecasts.

**Holt-Winters multiplicative** (RMSE 23.6) is competitive with the with-COVID SARIMAX (22.4), confirming that multiplicative seasonality is the dominant structural signal. But the COVID-excluded SARIMAX with exogenous clearly beats it (12.9 vs 23.6).

**All structured models crush the baselines** — seasonal naive sits at RMSE 78.2, last value at 91.7.

Note: all models are evaluated on the same 24-month holdout. In a stricter setup, model selection would use a rolling-origin CV (§8) and the holdout would only evaluate the winning model. The current approach uses the holdout for both comparison and evaluation, which slightly inflates optimism for the best model.

## 6. Exogenous coefficient inspection

The model comparison shows whether exogenous variables help in aggregate. This section inspects *which* variables carry the signal, by extracting the estimated coefficients from both SARIMAX specifications.

In [ ]:
def extract_exog_coefs(fit_result, exog_cols, label):
    rows = []
    for col in exog_cols:
        if col in fit_result.params.index:
            rows.append({
                "variable": col,
                "coef": round(fit_result.params[col], 4),
                "std_err": round(fit_result.bse[col], 4),
                "z_stat": round(fit_result.zvalues[col], 2),
                "p_value": round(fit_result.pvalues[col], 4),
            })
    out = pd.DataFrame(rows)
    out.insert(0, "model", label)
    return out

coef_full = extract_exog_coefs(fitted_models["SARIMAX with COVID + exog"], exog_cols, "with COVID")
coef_excl = extract_exog_coefs(fitted_models["SARIMAX excl. COVID + exog"], exog_cols, "excl. COVID")

display(pd.concat([coef_full, coef_excl], ignore_index=True))

The coefficient inspection confirms the OLS screening from §3.7, with one important addition.

In the **with-COVID model**, two variables are significant: **shock_dummy** (coef = −79.9, z = −21.4, p < 0.001) captures the COVID collapse, and **weather_severity** (coef = 5.4, z = 2.4, p = 0.016) has a modest but real effect. The remaining variables — search interest (p = 0.27), ticket price (p = 0.22), strike (p = 0.33) — are non-significant.

In the **excl. COVID model**, no variable reaches significance — the largest z-stat is ticket_price at 1.13 (p = 0.26). Yet the holdout comparison shows that the exogenous variables collectively improve RMSE from 17.6 to 12.9. This apparent contradiction — individually non-significant but collectively useful — is consistent with a diffuse signal spread across correlated covariates, where no single variable is strong enough to isolate statistically but the combined information tightens forecasts.

The practical implication: including the exogenous variables is justified by their holdout performance, not by their individual significance.

## 7. Residual diagnostics

Before trusting the holdout results, I check whether the model residuals satisfy the SARIMAX assumptions: no residual autocorrelation (Ljung-Box), approximate normality (QQ plot), and no systematic bias (residual mean). I run diagnostics on the primary model (SARIMAX excl. COVID).

In [ ]:
residuals = fitted_models["SARIMAX excl. COVID + exog"].resid

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0,0].plot(train_no_covid.index, residuals.values)
axes[0,0].axhline(0, color="black", lw=0.5)
axes[0,0].set_title("Residuals over time")

axes[0,1].hist(residuals.dropna().values, bins=25, edgecolor="white")
axes[0,1].set_title("Residual distribution")

plot_acf(residuals.dropna(), lags=24, ax=axes[1,0])
sm.qqplot(residuals.dropna(), line="s", ax=axes[1,1])
plt.tight_layout()
plt.show()

lb = acorr_ljungbox(residuals.dropna(), lags=[6, 12, 18, 24], return_df=True)
display(lb)

print("Residual mean:", round(residuals.mean(), 3))
print("Residual std:", round(residuals.std(), 2))

The residual diagnostics for the primary model (SARIMAX excl. COVID + exog) are broadly acceptable. Ljung-Box passes comfortably at lags 6 (p = 0.95), 18 (p = 0.17), and 24 (p = 0.39), but is marginally significant at lag 12 (p = 0.043) — suggesting some residual seasonal autocorrelation that the (2,1,0)(0,1,1,12) specification does not fully capture.

The residual mean is essentially zero (−0.097), indicating no systematic bias. The residual std of 34.5 is moderate relative to the series mean (~350), reflecting mostly the COVID shock period. The QQ plot shows moderate tail departures driven by these same outliers, which affects confidence interval coverage but not point forecast consistency.

## 8. Rolling-origin evaluation

I evaluate the primary model (SARIMAX excl. COVID) on 12-month-ahead rolling forecasts across the full dataset (train + holdout), starting from month 48 and stepping every 12 months. Each fold uses only data up to the cutoff for training — there is no leakage.

In [ ]:
full_df = pd.concat([train_df, final_holdout_df])

rolling_rows = []
for cutoff in range(48, len(full_df) - 12, 12):
    sub_train = full_df.iloc[:cutoff]
    sub_test = full_df.iloc[cutoff:cutoff + 12]

    mask = (sub_train.index >= "2020-03-01") & (sub_train.index <= "2021-06-01")
    sub_clean = sub_train[~mask] if (~mask).sum() > 24 else sub_train

    row = {"cutoff": sub_train.index.max().strftime("%Y-%m")}

    try:
        endog = sub_clean["passengers_thousands"]
        exog_tr = sub_clean[exog_cols]
        exog_te = sub_test[exog_cols]

        mod = sm.tsa.statespace.SARIMAX(
            endog, exog=exog_tr,
            order=auto_no_covid.order, seasonal_order=auto_no_covid.seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        pred = mod.fit(disp=False, maxiter=200, method="lbfgs").get_forecast(
            steps=12, exog=exog_te).predicted_mean.values
        row["sarimax_excl_covid"] = round(rmse(sub_test["passengers_thousands"], pred), 1)
    except (np.linalg.LinAlgError, ValueError) as e:
        row["sarimax_excl_covid"] = np.nan
        print(f"  ⚠️ SARIMAX failed at cutoff {row['cutoff']}: {e}")

    naive = full_df["passengers_thousands"].shift(12).loc[sub_test.index].values
    row["seasonal_naive"] = round(rmse(sub_test["passengers_thousands"], naive), 1)

    rolling_rows.append(row)

display(pd.DataFrame(rolling_rows))

The rolling-origin evaluation confirms the holdout findings across multiple time windows. The SARIMAX excl. COVID achieves stable RMSE of **12–18** on pre-COVID folds (2017-12, 2018-12) and post-recovery folds (2022-12, 2023-12), consistently beating the seasonal naive by a wide margin.

The three problem folds are 2019-12 (RMSE 58.7 — forecasts directly into the COVID crash, which no model could anticipate), 2020-12 (RMSE 64.5), and 2021-12 (RMSE 53.9). On these folds, the model excludes COVID from training so it has no basis to forecast through the recovery. Notably, the seasonal naive also struggles on 2020-12 (RMSE 111.3) but is competitive on 2019-12 (41.0 vs 58.7) — it simply repeats the pre-crash pattern, which happens to be a reasonable guess when the crash hasn't happened yet.

## 9. Forecast accuracy significance

With only 24 holdout observations, a formal Diebold-Mariano test has limited power. Instead, I use a paired sign test on forecast errors: for each holdout month, which model has the smaller absolute error? If one model wins consistently, that is informative even without distributional assumptions.

In [ ]:
from scipy.stats import binomtest

errors = {}
for name, pred in predictions.items():
    if name in ["last_value"]:
        continue
    errors[name] = np.abs(final_holdout_df["passengers_thousands"].values - pred)

comparisons = [
    ("SARIMAX excl. COVID + exog", "seasonal_naive"),
    ("SARIMAX excl. COVID + exog", "SARIMAX with COVID + exog"),
    ("SARIMAX excl. COVID + exog", "Holt-Winters"),
    ("SARIMAX excl. COVID + exog", "SARIMAX excl. COVID (no exog)"),
    ("SARIMAX with COVID + exog", "SARIMAX with COVID (no exog)"),
    ("SARIMAX with COVID + exog", "seasonal_naive"),
]

for a, b in comparisons:
    wins_a = int((errors[a] < errors[b]).sum())
    n = int((errors[a] != errors[b]).sum())
    p = binomtest(wins_a, n, 0.5, alternative="greater").pvalue
    print(f"{a} vs {b}: {wins_a}/{n} months (p={p:.4f})")

The sign tests provide non-parametric confirmation of the holdout results:

- **SARIMAX excl. COVID + exog vs seasonal naive: 24/24 months (p < 0.0001)** — the advantage is ubiquitous, not driven by outliers.
- **SARIMAX excl. COVID + exog vs with COVID + exog: 17/24 (p = 0.032)** — excluding COVID is a significant improvement, though not on every month.
- **SARIMAX excl. COVID + exog vs Holt-Winters: 15/24 (p = 0.154)** — despite much better RMSE (12.9 vs 23.6), the SARIMAX does not significantly dominate month-by-month. The SARIMAX wins on magnitude (when it wins, it wins big), but HW is competitive on many individual months.
- **SARIMAX excl. COVID + exog vs excl. COVID no exog: 19/24 (p = 0.003)** — the exogenous variables provide a significant and consistent improvement, confirming that the collective signal is real even though no individual variable reaches significance.
- **SARIMAX with COVID + exog vs with COVID no exog: 12/24 (p = 0.58)** — exogenous variables provide zero value when COVID is in the training data, likely because the shock_dummy absorbs most of the signal and the remaining covariates are noise.

## Final remarks

All structured models substantially outperform the seasonal naive (RMSE 78 → 12–23). The best holdout model is the **SARIMAX trained excluding COVID with exogenous variables** (RMSE 12.9, MAE 10.3), beating the with-COVID SARIMAX (22.4) on 17/24 months (p = 0.032) and the seasonal naive on all 24 months.

Key findings:

- **Excluding COVID is the single biggest improvement on post-recovery data** (RMSE 22.4 → 12.9). But the rolling-origin reveals a nuance: the COVID-excluded model performs poorly on the immediate post-COVID folds (RMSE 54–65) where the shock_dummy is needed to model the recovery. The optimal production strategy would be to include COVID during the recovery phase and drop it once the series returns to baseline.

- **Exogenous variables help — but only in the clean (excl. COVID) model.** The with/without exogenous comparison is decisive: excl. COVID sees RMSE drop from 17.6 to 12.9 (p = 0.003 by sign test), while with-COVID shows no difference (22.4 vs 22.4, p = 0.58). The coefficient inspection reveals no individually significant variable in the excl. COVID model — the signal is diffuse but collectively real.

- **Multiplicative seasonality is the dominant structural signal.** The decomposition confirmed it (§3.2), and Holt-Winters exploits it natively (RMSE 23.6). The additive SARIMAX leaves value on the table — a log-transform would bridge this gap (not tested here due to time constraints).

- **auto_arima selects different orders on full vs COVID-excluded data** — (0,1,1)(1,1,0,12) vs (2,1,0)(0,1,1,12) — confirming that COVID changes the model structure, not just the parameters.

**What I would do next:**
- **Fit SARIMAX on log-transformed passengers** to handle multiplicative seasonality natively — highest-priority extension
- **Sensitivity analysis on the COVID exclusion window** to verify the result is not an artefact of the specific cutoff
- **Model the COVID recovery as a ramp** rather than a binary dummy
- **Use rolling-origin CV for model selection** and reserve the holdout strictly for the winning model — the current approach uses the holdout for both selection and evaluation
- **Build an exogenous forecast pipeline** — in production, the SARIMAX requires future values of exogenous variables, which introduces a second forecasting problem